# IRA Annual Report — Data Extraction Notebook
**Purpose:** Extract county-level gross direct premium income and national fire/property claim ratios
from the IRA 2023 and 2024 Annual Insurance Reports, then build the actuarial validation dataset
for the Housing Financial Vulnerability Score (HFVS) dissertation.

**Outputs produced:**
- `ira_county_premium.csv` — gross direct premium by county × year
- `ira_national_ratios.csv` — national fire/domestic package claim ratios
- `ira_validation_dataset.csv` — merged HFVS + insurance density dataset ready for correlation
- `phase8_ira_validation.png` — replacement scatter plot (real data, not placeholders)

**Before running:** Ensure both PDFs are at:
```
/content/drive/MyDrive/KHS_Dissertation/data/raw/Insurance_Industry_Annual_Report_2023.pdf
/content/drive/MyDrive/KHS_Dissertation/data/raw/Insurance_Industry_Annual_Report_2024.pdf
```


## 0 — Mount Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
pkgs = ['pdfplumber', 'pypdf', 'tabulate']
for p in pkgs:
    subprocess.run(['pip', 'install', p, '-q'], check=True)
print("✓ Dependencies ready")


## 1 — Imports & paths

In [ ]:
import pdfplumber, re, json
from pypdf import PdfReader
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE = Path('/content/drive/MyDrive/KHS_Dissertation')
RAW   = DRIVE / 'data' / 'raw'
OUT   = DRIVE / 'outputs'
FIGS  = OUT / 'figures'
TABS  = OUT / 'tables'
for p in [FIGS, TABS]: p.mkdir(parents=True, exist_ok=True)

PDF_2023 = RAW / 'Insurance_Industry_Annual_Report_2023.pdf'
PDF_2024 = RAW / 'Insurance_Industry_Annual_Report_2024.pdf'

for f in [PDF_2023, PDF_2024]:
    assert f.exists(), f"❌ PDF not found: {f}"
    print(f"✓ Found: {f.name}")

# Palette (matches dissertation style)
TEAL = '#00695C'; RED = '#B71C1C'; AMBER = '#E65100'; BLUE = '#1565C0'; GRAY = '#546E7A'

# All 47 county names (canonical spelling for merging)
ALL_COUNTIES = [
    'Mombasa','Kwale','Kilifi','Tana River','Lamu','Taita-Taveta',
    'Garissa','Wajir','Mandera','Marsabit','Isiolo','Meru',
    'Tharaka-Nithi','Embu','Kitui','Machakos','Makueni','Nyandarua',
    'Nyeri','Kirinyaga',"Murang'a",'Kiambu','Turkana','West Pokot',
    'Samburu','Trans Nzoia','Uasin Gishu','Elgeyo-Marakwet','Nandi',
    'Baringo','Laikipia','Nakuru','Narok','Kajiado','Kericho','Bomet',
    'Kakamega','Vihiga','Bungoma','Busia','Siaya','Kisumu',
    'Homa Bay','Migori','Kisii','Nyamira','Nairobi',
]
print(f"\n✓ Canonical county list: {len(ALL_COUNTIES)} counties")


## 2 — PDF inventory
Inspect both PDFs to confirm page count and that a text layer exists.
We also scan for the keyword 'Table 5' and 'Table 28' to find the right pages.


In [ ]:
def pdf_inventory(pdf_path):
    """Return page count, font status sample, and keyword page index."""
    reader = PdfReader(str(pdf_path))
    n = len(reader.pages)
    # Sample first page text
    sample = reader.pages[0].extract_text()[:200] if n else ''
    
    # Find pages containing key tables
    hits = {}
    with pdfplumber.open(str(pdf_path)) as pdf:
        for i, page in enumerate(pdf.pages):
            txt = (page.extract_text() or '')
            for kw in ['Table 5', 'Table 28', 'Table 22',
                       'Gross Direct Premium', 'Loss Ratio', 'Claim Ratio',
                       'Fire', 'Domestic Package']:
                if kw.lower() in txt.lower():
                    hits.setdefault(kw, []).append(i + 1)  # 1-based
    return {'pages': n, 'sample': sample, 'keyword_pages': hits}

print("=== 2023 Report ===")
inv23 = pdf_inventory(PDF_2023)
print(f"  Pages: {inv23['pages']}")
for kw, pgs in inv23['keyword_pages'].items():
    print(f"  '{kw}' on pages: {pgs[:8]}")

print("\n=== 2024 Report ===")
inv24 = pdf_inventory(PDF_2024)
print(f"  Pages: {inv24['pages']}")
for kw, pgs in inv24['keyword_pages'].items():
    print(f"  '{kw}' on pages: {pgs[:8]}")


## 3 — Visually inspect candidate table pages
Rasterise the pages identified above so we can confirm which page holds
Table 5 (county premium) and Table 28 (claim ratios) before attempting extraction.


In [ ]:
import subprocess, glob
from IPython.display import Image, display

def show_page(pdf_path, page_num, dpi=120, label=''):
    """Rasterise a single PDF page and display inline."""
    out_prefix = f'/tmp/ira_page_{label}_{page_num}'
    subprocess.run([
        'pdftoppm', '-jpeg', '-r', str(dpi),
        '-f', str(page_num), '-l', str(page_num),
        str(pdf_path), out_prefix
    ], check=True, capture_output=True)
    imgs = sorted(glob.glob(f'{out_prefix}*.jpg'))
    if imgs:
        print(f"Page {page_num} of {label}:")
        display(Image(filename=imgs[0], width=750))
    else:
        print(f"  ⚠️  No image produced for page {page_num}")

# Show first hit for Table 5 and Table 28 in 2023 report
# Adjust page numbers here if the inventory above shows different pages
t5_pages_23  = inv23['keyword_pages'].get('Table 5', [])
t28_pages_23 = inv23['keyword_pages'].get('Table 28', [])
t5_pages_24  = inv24['keyword_pages'].get('Table 5', [])
t28_pages_24 = inv24['keyword_pages'].get('Table 28', [])

print("Showing first Table 5 page for each report:")
if t5_pages_23:  show_page(PDF_2023, t5_pages_23[0], label='2023')
if t5_pages_24:  show_page(PDF_2024, t5_pages_24[0], label='2024')

print("\nShowing first Table 28 / claim ratio page:")
if t28_pages_23: show_page(PDF_2023, t28_pages_23[0], label='2023_claimratio')
if t28_pages_24: show_page(PDF_2024, t28_pages_24[0], label='2024_claimratio')


## 4 — Extract Table 5: Gross Direct Premium Income by County
`pdfplumber` table extraction with a fallback regex line parser.
The function returns a raw DataFrame; we clean and canonicalise county names afterwards.


In [ ]:
# ─── Canonical name harmoniser ────────────────────────────────────────────────
# IRA may spell some counties differently from your COUNTY_MAP. Add entries as needed.
NAME_MAP = {
    'muranga'           : "Murang'a",
    "murang'a"          : "Murang'a",
    'murang a'          : "Murang'a",
    'taita taveta'      : 'Taita-Taveta',
    'tana river'        : 'Tana River',
    'west pokot'        : 'West Pokot',
    'trans nzoia'       : 'Trans Nzoia',
    'uasin gishu'       : 'Uasin Gishu',
    'elgeyo marakwet'   : 'Elgeyo-Marakwet',
    'elgeyo-marakwet'   : 'Elgeyo-Marakwet',
    'homa bay'          : 'Homa Bay',
    'tharaka nithi'     : 'Tharaka-Nithi',
    'tharaka-nithi'     : 'Tharaka-Nithi',
    'nyandarua'         : 'Nyandarua',
    'nairobi city'      : 'Nairobi',
    'nairobi county'    : 'Nairobi',
    'mombasa'           : 'Mombasa',
}

def canonicalise(name: str) -> str:
    """Normalise a county name string to the dissertation canonical spelling."""
    if not isinstance(name, str):
        return ''
    cleaned = re.sub(r'\s+', ' ', name.strip().lower())
    return NAME_MAP.get(cleaned, name.strip().title())


def extract_table5_pdfplumber(pdf_path, year: int) -> pd.DataFrame:
    """
    Extract county × premium rows from Table 5.
    Strategy:
      1. Find pages flagged as containing 'Table 5' or 'Gross Direct Premium'.
      2. Use pdfplumber table extraction on those pages.
      3. If table extraction fails, fall back to regex on raw text.
    Returns DataFrame with columns: county_name, gdpi_{year}
    """
    rows = []
    
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            txt = page.extract_text() or ''
            
            # Only process pages that look like Table 5
            if not any(kw in txt for kw in
                       ['Gross Direct Premium', 'GROSS DIRECT PREMIUM', 'Table 5']):
                continue
            
            # ── Attempt 1: structured table extraction ────────────────────
            tables = page.extract_tables({
                'vertical_strategy'   : 'lines',
                'horizontal_strategy' : 'lines',
                'snap_tolerance'      : 3,
            })
            
            for tbl in (tables or []):
                for row in tbl:
                    if not row or len(row) < 2:
                        continue
                    name_cell = str(row[0] or '').strip()
                    # Last non-empty cell is the total premium
                    vals = [c for c in row[1:] if c and str(c).strip()]
                    if not vals:
                        continue
                    raw_val = str(vals[-1]).replace(',', '').replace(' ', '').strip()
                    try:
                        premium = float(raw_val)
                    except ValueError:
                        continue
                    if premium <= 0:
                        continue
                    county = canonicalise(name_cell)
                    if county in ALL_COUNTIES:
                        rows.append({'county_name': county, f'gdpi_{year}': premium})
            
            # ── Attempt 2: regex line parser (fallback if table empty) ────
            if not rows:
                for line in txt.split('\n'):
                    # Pattern: county name followed by numbers, last being total
                    nums = re.findall(r'[\d,]+\.?\d*', line)
                    if not nums:
                        continue
                    # County name is everything before the first number
                    name_part = re.split(r'[\d,]', line)[0].strip()
                    if len(name_part) < 3:
                        continue
                    county = canonicalise(name_part)
                    if county not in ALL_COUNTIES:
                        continue
                    # Use the last numeric token as total premium
                    raw_val = nums[-1].replace(',', '')
                    try:
                        premium = float(raw_val)
                    except ValueError:
                        continue
                    if premium > 0:
                        rows.append({'county_name': county, f'gdpi_{year}': premium})
    
    df = pd.DataFrame(rows).drop_duplicates('county_name')
    print(f"  [{year}] Extracted {len(df)} counties from Table 5")
    if len(df) < 47:
        missing = set(ALL_COUNTIES) - set(df['county_name'])
        print(f"  ⚠️  Missing ({len(missing)}): {sorted(missing)}")
    return df


print("Extracting 2023 county premium data...")
df_prem_23 = extract_table5_pdfplumber(PDF_2023, 2023)

print("\nExtracting 2024 county premium data...")
df_prem_24 = extract_table5_pdfplumber(PDF_2024, 2024)


### 4b — Manual override for any missing counties
If the automated extractor missed some counties (the table may span multiple pages
or use a non-standard layout), paste the missing values here.
Run the cell above first to see which counties are missing, then fill in below.


In [ ]:
# ── Paste missing 2023 values here (KES thousands as reported in Table 5) ─────
# Format: 'County Name': value_in_KES
MANUAL_2023 = {
    # Example: 'Lamu': 45_678_000,
    # 'Marsabit': 23_100_000,
}

# ── Paste missing 2024 values here ────────────────────────────────────────────
MANUAL_2024 = {
    # Example: 'Lamu': 51_000_000,
}

# Apply overrides
if MANUAL_2023:
    override_23 = pd.DataFrame([
        {'county_name': k, 'gdpi_2023': v} for k, v in MANUAL_2023.items()
    ])
    df_prem_23 = pd.concat([
        df_prem_23[~df_prem_23['county_name'].isin(override_23['county_name'])],
        override_23
    ], ignore_index=True)
    print(f"Applied {len(MANUAL_2023)} manual 2023 overrides")

if MANUAL_2024:
    override_24 = pd.DataFrame([
        {'county_name': k, 'gdpi_2024': v} for k, v in MANUAL_2024.items()
    ])
    df_prem_24 = pd.concat([
        df_prem_24[~df_prem_24['county_name'].isin(override_24['county_name'])],
        override_24
    ], ignore_index=True)
    print(f"Applied {len(MANUAL_2024)} manual 2024 overrides")

print(f"\nFinal county counts — 2023: {len(df_prem_23)}, 2024: {len(df_prem_24)}")


## 5 — Extract Table 28: National Claim Ratios
The IRA reports class-level loss/claim ratios **nationally** (not per county).
We extract the Fire class and Domestic Package class ratios — the closest proxies
to residential property insurance — to use as a calibration anchor.


In [ ]:
def extract_claim_ratios(pdf_path, year: int) -> dict:
    """
    Extract net incurred claim ratios from Table 28 (or equivalent table).
    Target classes: Fire, Domestic Package, Engineering, Liability (for context).
    Returns a dict: {class_name: ratio_as_float}
    """
    target_classes = {
        'fire'             : 'Fire',
        'domestic package' : 'Domestic Package',
        'domestic'         : 'Domestic Package',
        'engineering'      : 'Engineering',
        'liability'        : 'Liability',
        'motor'            : 'Motor',         # include for context / sanity check
        'total'            : 'Total General Insurance',
    }
    ratios = {}
    
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            txt = page.extract_text() or ''
            
            # Only process pages that look like claim ratio tables
            if not any(kw in txt.lower() for kw in
                       ['claim ratio', 'loss ratio', 'incurred', 'table 28']):
                continue
            
            # Structured extraction first
            tables = page.extract_tables({
                'vertical_strategy'   : 'lines',
                'horizontal_strategy' : 'lines',
            })
            for tbl in (tables or []):
                for row in tbl:
                    if not row or len(row) < 2:
                        continue
                    label = str(row[0] or '').strip().lower()
                    for key, canonical in target_classes.items():
                        if key in label:
                            # Find numeric value (percentage — could be 0.45 or 45.0)
                            for cell in row[1:]:
                                raw = str(cell or '').replace(',','').replace('%','').strip()
                                try:
                                    val = float(raw)
                                    # Normalise: if > 1, assume percentage points
                                    if val > 1:
                                        val = val / 100
                                    if 0 < val < 2:  # sanity: claim ratio between 0% and 200%
                                        ratios[canonical] = val
                                        break
                                except ValueError:
                                    continue
            
            # Fallback: regex on lines
            for line in txt.split('\n'):
                for key, canonical in target_classes.items():
                    if key in line.lower() and canonical not in ratios:
                        nums = re.findall(r'\d+\.?\d*', line)
                        for n in nums:
                            val = float(n)
                            if val > 1:
                                val = val / 100
                            if 0 < val < 2:
                                ratios[canonical] = val
                                break
    
    print(f"  [{year}] Extracted claim ratios:")
    for cls, ratio in ratios.items():
        print(f"    {cls}: {ratio:.3f} ({ratio*100:.1f}%)")
    return ratios


print("Extracting 2023 national claim ratios...")
ratios_2023 = extract_claim_ratios(PDF_2023, 2023)

print("\nExtracting 2024 national claim ratios...")
ratios_2024 = extract_claim_ratios(PDF_2024, 2024)

# Build tidy DataFrame
ratio_rows = []
for cls, r in ratios_2023.items():
    ratio_rows.append({'insurance_class': cls, 'year': 2023, 'claim_ratio': r})
for cls, r in ratios_2024.items():
    ratio_rows.append({'insurance_class': cls, 'year': 2024, 'claim_ratio': r})

df_ratios = pd.DataFrame(ratio_rows)
print("\nNational claim ratios table:")
print(df_ratios.to_string(index=False))

# Save
df_ratios.to_csv(TABS / 'ira_national_ratios.csv', index=False)
print(f"\n✓ Saved: {TABS / 'ira_national_ratios.csv'}")


### 5b — Manual override for claim ratios
If the automated extractor missed values (common when the table uses complex
merged cells), paste them here. Values should be the decimal form (e.g. 0.48 for 48%).


In [ ]:
# ── Fill these in from the PDF if the extractor missed them ───────────────────
# Use the decimal form: 48% → 0.48
MANUAL_RATIOS = {
    # 2023: {'Fire': 0.XX, 'Domestic Package': 0.XX}
    # 2024: {'Fire': 0.XX, 'Domestic Package': 0.XX}
    2023: {
        # 'Fire': 0.48,
        # 'Domestic Package': 0.42,
    },
    2024: {
        # 'Fire': 0.51,
        # 'Domestic Package': 0.45,
    },
}

# Apply overrides and re-save
for yr, cls_map in MANUAL_RATIOS.items():
    for cls, ratio in cls_map.items():
        mask = (df_ratios['year'] == yr) & (df_ratios['insurance_class'] == cls)
        if mask.any():
            df_ratios.loc[mask, 'claim_ratio'] = ratio
            print(f"  Updated [{yr}] {cls} → {ratio:.3f}")
        else:
            new_row = pd.DataFrame([{'insurance_class': cls, 'year': yr, 'claim_ratio': ratio}])
            df_ratios = pd.concat([df_ratios, new_row], ignore_index=True)
            print(f"  Added   [{yr}] {cls} = {ratio:.3f}")

df_ratios.to_csv(TABS / 'ira_national_ratios.csv', index=False)
print("\n✓ Manual ratios applied and saved.")


## 6 — Build county premium master table
Merge 2023 and 2024 premium data, compute year-on-year growth, and save.


In [ ]:
# Outer merge to keep all counties present in either year
df_prem = df_prem_23.merge(df_prem_24, on='county_name', how='outer')

# Growth rate
df_prem['premium_growth_pct'] = (
    (df_prem['gdpi_2024'] - df_prem['gdpi_2023']) / df_prem['gdpi_2023'] * 100
)

# National totals for share calculation
nat_total_23 = df_prem['gdpi_2023'].sum()
nat_total_24 = df_prem['gdpi_2024'].sum()
df_prem['premium_share_2023_pct'] = df_prem['gdpi_2023'] / nat_total_23 * 100
df_prem['premium_share_2024_pct'] = df_prem['gdpi_2024'] / nat_total_24 * 100

# Sort by 2023 premium descending
df_prem = df_prem.sort_values('gdpi_2023', ascending=False).reset_index(drop=True)

print(f"County premium master: {len(df_prem)} counties")
print(f"National GDPI 2023: KES {nat_total_23:,.0f}")
print(f"National GDPI 2024: KES {nat_total_24:,.0f}")
print(f"\nTop 10 counties by 2023 premium:")
print(df_prem[['county_name','gdpi_2023','premium_share_2023_pct']].head(10).to_string(index=False))

# Concentration index: top-5 share vs bottom-10 share
top5_share = df_prem.head(5)['premium_share_2023_pct'].sum()
bot10_share = df_prem.tail(10)['premium_share_2023_pct'].sum()
print(f"\nPremium concentration — Top 5 counties: {top5_share:.1f}% of national total")
print(f"Premium concentration — Bottom 10 counties: {bot10_share:.1f}% of national total")

df_prem.to_csv(TABS / 'ira_county_premium.csv', index=False)
print(f"\n✓ Saved: {TABS / 'ira_county_premium.csv'}")


## 7 — Merge with HFVS county_risk table
Load the `county_risk` DataFrame produced in Phase 6 and merge with IRA premium data
to compute insurance density (premium per surveyed household) per county.


In [ ]:
import os, sys
sys.path.insert(0, '/content/KHS_housing_dissertation/src')

# Load county_risk (produced by Phase 6 of the main dissertation notebook)
# Expected columns: county_name, mean_hfvs, pct_urban, pct_triple_exposed, n_households
PARQ = DRIVE / 'data' / 'parquet'
county_risk_path = PARQ / 'county_risk.parquet'

if county_risk_path.exists():
    county_risk = pd.read_parquet(county_risk_path)
    print(f"✓ Loaded county_risk: {county_risk.shape}")
    print(county_risk.columns.tolist())
else:
    # Fallback: try loading from outputs/tables
    alt = TABS / 'county_risk.csv'
    if alt.exists():
        county_risk = pd.read_csv(alt)
        print(f"✓ Loaded county_risk from CSV: {county_risk.shape}")
    else:
        raise FileNotFoundError(
            "county_risk not found. Run Phase 6 of the main notebook first, "
            "or ensure county_risk.parquet exists in data/parquet/."
        )

# ── Merge ─────────────────────────────────────────────────────────────────────
ira_val = county_risk.merge(df_prem, on='county_name', how='inner')
print(f"\nMatched {len(ira_val)} counties after merge")
print(f"Unmatched in county_risk: {set(county_risk['county_name']) - set(ira_val['county_name'])}")

# Insurance density: GDPI per surveyed household
# Use 'n_households' from county_risk (KHS sample size per county)
hh_col = 'n_households' if 'n_households' in ira_val.columns else 'n'
ira_val['insurance_density_2023'] = ira_val['gdpi_2023'] / ira_val[hh_col]
ira_val['insurance_density_2024'] = ira_val['gdpi_2024'] / ira_val[hh_col]

# Log-transform for the scatter (insurance density is heavily right-skewed)
ira_val['log_ins_density_2023'] = np.log1p(ira_val['insurance_density_2023'])

print(f"\nInsurance density summary (KES per surveyed household, 2023):")
print(ira_val['insurance_density_2023'].describe().round(0))


## 8 — Three-part actuarial validation
Run the three Spearman correlations described in the dissertation methodology.


In [ ]:
# ── Correlation 1: HFVS vs insurance density (main validation) ────────────────
rho1, p1 = stats.spearmanr(ira_val['mean_hfvs'], ira_val['insurance_density_2023'])
print(f"[1] HFVS vs Insurance Density 2023:")
print(f"    Spearman rho = {rho1:.3f}  (p = {p1:.4f})")
print(f"    Interpretation: {'✓ Expected negative relationship (high vuln → low density)'  if rho1 < 0 else '⚠️ Positive — investigate outliers'}")

# ── Correlation 2: HFVS vs premium growth ─────────────────────────────────────
valid_growth = ira_val.dropna(subset=['premium_growth_pct', 'mean_hfvs'])
rho2, p2 = stats.spearmanr(valid_growth['mean_hfvs'], valid_growth['premium_growth_pct'])
print(f"\n[2] HFVS vs Premium Growth 2023–24:")
print(f"    Spearman rho = {rho2:.3f}  (p = {p2:.4f})")

# ── Correlation 3: Triple exposure vs insurance density ───────────────────────
if 'pct_triple_exposed' in ira_val.columns:
    rho3, p3 = stats.spearmanr(ira_val['pct_triple_exposed'], ira_val['insurance_density_2023'])
    print(f"\n[3] Triple Exposure vs Insurance Density 2023:")
    print(f"    Spearman rho = {rho3:.3f}  (p = {p3:.4f})")
else:
    print("\n[3] pct_triple_exposed column not found — skipping")

# ── National fire claim ratio anchor ──────────────────────────────────────────
fire_ratio_23 = df_ratios.query("insurance_class == 'Fire' and year == 2023")['claim_ratio']
fire_ratio_val = float(fire_ratio_23.iloc[0]) if len(fire_ratio_23) else None

if fire_ratio_val:
    # Vulnerability loading from regression coefficient of Correlation 1
    from sklearn.linear_model import LinearRegression
    X = ira_val[['mean_hfvs']].values
    y = ira_val['insurance_density_2023'].values
    valid_mask = ~np.isnan(X.ravel()) & ~np.isnan(y)
    lr = LinearRegression().fit(X[valid_mask], y[valid_mask])
    
    # High-vulnerability counties (HFVS > 0.60)
    high_vuln = ira_val[ira_val['mean_hfvs'] > 0.60]
    avg_hfvs_high = high_vuln['mean_hfvs'].mean()
    
    # Expected insurance density loading (illustrative)
    loading_factor = 1 + abs(rho1) * 0.5  # conservative loading proxy
    expected_ratio_high_vuln = fire_ratio_val * loading_factor
    
    print(f"\n[Calibration Anchor]")
    print(f"  National Fire claim ratio (2023): {fire_ratio_val:.3f} ({fire_ratio_val*100:.1f}%)")
    print(f"  High-vulnerability counties (HFVS > 0.60): {len(high_vuln)}")
    print(f"  Expected fire claim ratio (high-vuln, loaded): "
          f"{expected_ratio_high_vuln:.3f} ({expected_ratio_high_vuln*100:.1f}%)")
    print(f"  → Actuarial pricing uplift implied: {(loading_factor-1)*100:.0f}%")


## 9 — Scatter plot: HFVS vs Insurance Density
Replaces the placeholder `phase8_ira_validation.png` with real IRA data.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6.5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#F8F8F6')
for sp in ['top','right']: ax.spines[sp].set_visible(False)

# Colour by urban share if available
colour_col = 'pct_urban' if 'pct_urban' in ira_val.columns else None
c_vals = ira_val[colour_col] if colour_col else BLUE

sc = ax.scatter(
    ira_val['mean_hfvs'],
    ira_val['insurance_density_2023'],
    c=c_vals, cmap='RdYlGn_r', s=80, alpha=0.85,
    edgecolors='white', linewidth=0.6, zorder=3
)

# County labels
for _, row in ira_val.iterrows():
    ax.annotate(
        row['county_name'][:8],
        (row['mean_hfvs'], row['insurance_density_2023']),
        fontsize=5.5, alpha=0.72, xytext=(2, 2), textcoords='offset points'
    )

# Regression line
valid = ira_val.dropna(subset=['mean_hfvs', 'insurance_density_2023'])
if len(valid) > 2:
    m, b = np.polyfit(valid['mean_hfvs'], valid['insurance_density_2023'], 1)
    xr = np.linspace(valid['mean_hfvs'].min(), valid['mean_hfvs'].max(), 60)
    ax.plot(xr, m*xr + b, color=RED, lw=1.8, ls='--', label='OLS trend', zorder=4)

if colour_col:
    plt.colorbar(sc, ax=ax, label='% Urban', shrink=0.75)

ax.set_xlabel('Mean HFVS (survey-weighted)', fontsize=11)
ax.set_ylabel('Insurance Density 2023\n(Gross Direct Premium / KHS households, KES)', fontsize=10)
ax.set_title(
    f'IRA Validation: HFVS vs County Insurance Density (2023)\n'
    f'Spearman ρ = {rho1:.3f}  (p = {p1:.4f})  |  n = {len(valid)} counties',
    fontsize=12, fontweight='600'
)

# Quadrant lines at medians
med_x = ira_val['mean_hfvs'].median()
med_y = ira_val['insurance_density_2023'].median()
ax.axvline(med_x, color=GRAY, lw=0.8, ls=':', alpha=0.6)
ax.axhline(med_y, color=GRAY, lw=0.8, ls=':', alpha=0.6)
ax.text(med_x + 0.002, ax.get_ylim()[1]*0.97, 'Median HFVS',
        fontsize=7, color=GRAY, va='top')

# Source note
fig.text(0.5, -0.03,
    'Source: IRA Insurance Industry Annual Report 2023, Table 5; KHS 2023/24 household survey.',
    ha='center', fontsize=8, color=GRAY, style='italic')

plt.legend(fontsize=8)
plt.tight_layout()
out_fig = FIGS / 'phase8_ira_validation.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Saved: {out_fig}")
print(f"   This file REPLACES the placeholder version from Phase 8.4")


## 10 — Premium concentration chart
Shows that insurance market depth follows wealth, not risk — a key dissertation finding.


In [ ]:
# Merge concentration data with HFVS for colouring
conc = df_prem.merge(
    county_risk[['county_name','mean_hfvs']].dropna(),
    on='county_name', how='left'
).sort_values('gdpi_2023', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#F8F8F6')
for sp in ['top','right']: ax.spines[sp].set_visible(False)

colours = [RED if h > 0.60 else TEAL if h < 0.40 else AMBER
           for h in conc['mean_hfvs'].fillna(0.5)]

bars = ax.barh(conc['county_name'], conc['premium_share_2023_pct'],
               color=colours, edgecolor='white', linewidth=0.4)
ax.invert_yaxis()
ax.set_xlabel('Share of National Gross Direct Premium, 2023 (%)', fontsize=10)
ax.set_title('Premium Concentration by County (Top 20, 2023)\n'
             'Red = High HFVS (>0.60) · Teal = Low HFVS (<0.40)',
             fontsize=11, fontweight='600')

for bar, (_, row) in zip(bars, conc.iterrows()):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{row['premium_share_2023_pct']:.1f}%",
            va='center', fontsize=7.5)

fig.text(0.5, -0.03,
    'Source: IRA Insurance Industry Annual Report 2023, Table 5.',
    ha='center', fontsize=8, color=GRAY, style='italic')
plt.tight_layout()
out_fig2 = FIGS / 'phase8_premium_concentration.png'
plt.savefig(out_fig2, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {out_fig2}")


## 11 — Save final validation dataset

In [ ]:
# Full validation dataset for use in Phase 8.7 quadrant chart
out_cols = [c for c in [
    'county_name', 'mean_hfvs', 'pct_urban', 'pct_triple_exposed',
    hh_col if 'hh_col' in dir() else 'n_households',
    'gdpi_2023', 'gdpi_2024', 'premium_growth_pct',
    'premium_share_2023_pct', 'premium_share_2024_pct',
    'insurance_density_2023', 'insurance_density_2024',
    'log_ins_density_2023',
] if c in ira_val.columns]

ira_val[out_cols].to_csv(TABS / 'ira_validation_dataset.csv', index=False)
print(f"✓ Saved: {TABS / 'ira_validation_dataset.csv'}")
print(f"   Columns: {out_cols}")
print(f"   Rows (counties): {len(ira_val)}")

# ── Summary report ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EXTRACTION SUMMARY")
print("="*60)
print(f"  Counties with 2023 premium data : {df_prem['gdpi_2023'].notna().sum()}")
print(f"  Counties with 2024 premium data : {df_prem['gdpi_2024'].notna().sum()}")
print(f"  National claim ratios extracted : {len(df_ratios)}")
print(f"  Validation Spearman rho (HFVS vs density): {rho1:.3f} (p={p1:.4f})")
print()
print("  Figures saved:")
print(f"    {FIGS}/phase8_ira_validation.png")
print(f"    {FIGS}/phase8_premium_concentration.png")
print()
print("  Tables saved:")
print(f"    {TABS}/ira_county_premium.csv")
print(f"    {TABS}/ira_national_ratios.csv")
print(f"    {TABS}/ira_validation_dataset.csv")
print()
print("  Citation template:")
print('  "County-level gross direct premium income was extracted from Table 5')
print('   of the IRA Insurance Industry Annual Reports for 2023 and 2024')
print('   (IRA, 2024; IRA, 2025). Insurance density was computed as county')
print('   gross direct premium divided by the number of households in the')
print('   2023/24 KHS sample for that county."')
